In [0]:
# Databricks notebook source
catalog = "meridian_t12"

# 1. Initialize ops logging table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.pipeline_quality_log (
    run_timestamp TIMESTAMP,
    check_name STRING,
    status STRING,
    details STRING
)
""")

# Support deliberate failure parameter for step 7
dbutils.widgets.text("fail_test", "false", "Force Failure")
fail_test = dbutils.widgets.get("fail_test").lower() == "true"

checks_passed = True
errors = []

# Check A: Verify Gold Table is not empty
gold_count = spark.table(f"{catalog}.gold.gold_fact_bookings").count()
if gold_count == 0 or fail_test:
    checks_passed = False
    msg = f"Empty Gold table: {gold_count} rows found (fail_test={fail_test})"
    errors.append(msg)
    spark.sql(f"INSERT INTO {catalog}.ops.pipeline_quality_log VALUES (current_timestamp(), 'EMPTY_GOLD_CHECK', 'FAIL', '{msg}')")
else:
    spark.sql(f"INSERT INTO {catalog}.ops.pipeline_quality_log VALUES (current_timestamp(), 'EMPTY_GOLD_CHECK', 'PASS', 'Rows: {gold_count}')")

# Check B: Duplicate Primary Keys
dups = spark.sql(f"""
    SELECT booking_id, COUNT(*) 
    FROM {catalog}.gold.gold_fact_bookings 
    GROUP BY booking_id 
    HAVING COUNT(*) > 1
""").count()

if dups > 0:
    checks_passed = False
    msg = f"Duplicate primary keys found: {dups}"
    errors.append(msg)
    spark.sql(f"INSERT INTO {catalog}.ops.pipeline_quality_log VALUES (current_timestamp(), 'DUPLICATE_KEY_CHECK', 'FAIL', '{msg}')")
else:
    spark.sql(f"INSERT INTO {catalog}.ops.pipeline_quality_log VALUES (current_timestamp(), 'DUPLICATE_KEY_CHECK', 'PASS', 'No duplicates')")

# Check C: Orphan Records
orphans = spark.sql(f"""
    SELECT b.booking_id 
    FROM {catalog}.gold.gold_fact_bookings b
    LEFT JOIN {catalog}.silver.silver_destinations d ON b.destination_id = d.destination_id
    WHERE b.destination_id IS NOT NULL AND d.destination_id IS NULL
""").count()

if orphans > 0:
    checks_passed = False
    msg = f"Orphan records found: {orphans}"
    errors.append(msg)
    spark.sql(f"INSERT INTO {catalog}.ops.pipeline_quality_log VALUES (current_timestamp(), 'ORPHAN_ROW_CHECK', 'FAIL', '{msg}')")
else:
    spark.sql(f"INSERT INTO {catalog}.ops.pipeline_quality_log VALUES (current_timestamp(), 'ORPHAN_ROW_CHECK', 'PASS', 'Zero orphans')")

if not checks_passed:
    raise ValueError(f"Quality Gate Failed: {' | '.join(errors)}")

print("All Quality Checks Passed.")